# Mistral-7B QLoRA Fine-Tuning
**LLM Platform — Week 3**

Runtime: GPU (T4 minimum, A100 recommended)  
Estimated time: ~45 min on T4, ~15 min on A100

### Steps
1. Install dependencies
2. Upload training data from your local machine
3. Run QLoRA fine-tuning with MLflow logging
4. Evaluate on test set
5. Download checkpoint

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────────
!pip install -q \
    torch>=2.1.0 \
    'transformers==4.46.*' \
    'peft==0.13.*' \
    'trl==0.12.*' \
    'bitsandbytes==0.44.*' \
    'accelerate==1.*' \
    'datasets==3.*' \
    'mlflow==2.18.*' \
    sentencepiece \
    protobuf

In [ ]:
# ── 2. Verify GPU ─────────────────────────────────────────────────────────────
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# ── 3. Upload training data ────────────────────────────────────────────────────
# Upload data/train_formatted.jsonl, data/val_formatted.jsonl, data/test_formatted.jsonl
# from your local machine using the Files panel on the left, or run:
#   from google.colab import files
#   uploaded = files.upload()  # then select the 3 files

import os
os.makedirs('data', exist_ok=True)

from google.colab import files
print('Upload train_formatted.jsonl, val_formatted.jsonl, test_formatted.jsonl')
uploaded = files.upload()
for name, content in uploaded.items():
    with open(f'data/{name}', 'wb') as f:
        f.write(content)
    print(f'Saved data/{name}')

In [ ]:
# ── 4. Hugging Face login (needed for Mistral gated model) ───────────────────
from huggingface_hub import login
# Paste your HF token here (read access is enough)
HF_TOKEN = 'hf_...'  # <-- replace
login(token=HF_TOKEN)

In [ ]:
# ── 5. Config ─────────────────────────────────────────────────────────────────
BASE_MODEL   = 'mistralai/Mistral-7B-Instruct-v0.3'
TRAIN_FILE   = 'data/train_formatted.jsonl'
VAL_FILE     = 'data/val_formatted.jsonl'
OUTPUT_DIR   = 'checkpoints/mistral-7b-lora-v1'
RUN_NAME     = 'mistral-7b-lora-v1'
EPOCHS       = 3
BATCH_SIZE   = 4
GRAD_ACCUM   = 4       # effective batch = 16
LR           = 2e-4
MAX_SEQ_LEN  = 2048
LORA_R       = 16
LORA_ALPHA   = 32

In [ ]:
# ── 6. Load model in 4-bit ────────────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
)
print('Model loaded')

In [ ]:
# ── 7. Apply LoRA ─────────────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(prepare_model_for_kbit_training(model), lora_config)
model.print_trainable_parameters()

In [ ]:
# ── 8. Load datasets ──────────────────────────────────────────────────────────
from datasets import load_dataset

train_ds = load_dataset('json', data_files=TRAIN_FILE, split='train')
val_ds   = load_dataset('json', data_files=VAL_FILE,   split='train')
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
# ── 9. Train ──────────────────────────────────────────────────────────────────
import mlflow
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    report_to='none',
)

mlflow.set_experiment('llm-platform-finetuning')
with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params({'r': LORA_R, 'alpha': LORA_ALPHA, 'lr': LR,
                       'epochs': EPOCHS, 'base_model': BASE_MODEL})

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
        args=training_args,
    )
    trainer.train()
    trainer.save_model(OUTPUT_DIR)

    RUN_ID = run.info.run_id
    print(f'MLflow run_id: {RUN_ID}')

In [ ]:
# ── 10. Quick test inference ───────────────────────────────────────────────────
from peft import PeftModel
from transformers import pipeline

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
)

prompt = '<s>[INST] What is LoRA and why is it efficient? [/INST]'
output = pipe(prompt)[0]['generated_text']
print(output[len(prompt):])

In [ ]:
# ── 11. Download checkpoint ────────────────────────────────────────────────────
import shutil
from google.colab import files

archive = shutil.make_archive('mistral-7b-lora-v1', 'zip', OUTPUT_DIR)
files.download(archive)
print('Checkpoint downloaded. Unzip and place in llm-platform/checkpoints/')

In [ ]:
# ── 12. Save MLflow run_id for local promotion check ─────────────────────────
import json
with open('mlflow_run_id.txt', 'w') as f:
    f.write(RUN_ID)
files.download('mlflow_run_id.txt')
print(f'Run ID: {RUN_ID}')
print('Keep this for: python scripts/promote_model.py --run_id <RUN_ID>')